# V3.2 — IAM Aachen (Scratch Training)

**Accelerator:** GPU T4 x1

**Değişiklikler (v3_augmented üzerine):**
- **B1:** Early stopping artık `val_loss` değil `val_wa` izliyor
  → v3_augmented ep51'de val_wa tüm zamanların en yükseğindeyken (89.61%) durdu
- **B2:** Epochs 60→80, patience 15→20

Augmentation aynı: elastic deformation + morphological ops.

**Tahmini süre:** ~3-3.5 saat (T4, 80 epoch / early stopping)

**Gerekli datasetslar (Add Data):**
1. `brht25/crnn-h2-code` (scripts + aachen_splits)
2. IAM dataset (`paper-traning-data`)

**Çıktı:** `results/v3_2_results.json` — Test WA, Wilson CI, McNemar p

---
**KURALLAR:** Test set'e sadece bu notebook'ta, bir kez bakılır. Cherry-pick yok.

---
**Donanım (makale için):**
- GPU: NVIDIA Tesla T4, 15360 MiB VRAM
- CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
- RAM: 13 GB
- Env: Kaggle Notebooks, Python 3.10, PyTorch 2.3.0+cu121

In [ ]:
# Hücre 1: GPU + donanım bilgisi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.0f} GB ({int(vram*1024)} MiB)")
    print(f"PyTorch: {torch.__version__}")

!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h | grep Mem
!python --version

In [ ]:
# Hücre 2: crnn-h2-code dataset'ten kopyala (base), sonra v3.2 override
import sys, os, shutil

CODE_INPUT = "/kaggle/input/datasets/brht25/crnn-h2-code"

if os.path.exists(CODE_INPUT):
    os.makedirs("/kaggle/working/cloud", exist_ok=True)

    # Base scripts (cloud/)
    for fname in os.listdir(f"{CODE_INPUT}/cloud"):
        if fname.endswith((".py", ".txt", ".sh")):
            shutil.copy(f"{CODE_INPUT}/cloud/{fname}", f"/kaggle/working/cloud/{fname}")

    # V3.2 override: model_v3.py (B1 fix) + v3_2_train.py (B2)
    v32_src = f"{CODE_INPUT}/cloud/v3.2"
    if os.path.exists(v32_src):
        os.makedirs("/kaggle/working/cloud/v3.2", exist_ok=True)
        for fname in os.listdir(v32_src):
            if fname.endswith((".py", ".txt")):
                shutil.copy(f"{v32_src}/{fname}", f"/kaggle/working/cloud/{fname}")
                shutil.copy(f"{v32_src}/{fname}", f"/kaggle/working/cloud/v3.2/{fname}")
        print("V3.2 override kopyalandı OK")
    else:
        print(f"⚠️  {v32_src} bulunamadı")

    shutil.copytree(f"{CODE_INPUT}/aachen_splits", "/kaggle/working/aachen_splits", dirs_exist_ok=True)
    shutil.copy(f"{CODE_INPUT}/trigram_lm.py", "/kaggle/working/trigram_lm.py")
    print("Scripts + aachen_splits kopyalandı OK")
else:
    print(f"⚠️  {CODE_INPUT} bulunamadı — Add Data → Your Datasets → crnn-h2-code")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
!pip install -q word-beam-search
!pip install -q -r cloud/requirements.txt

# WBS kurulum kontrolü
try:
    import word_beam_search
    print("✓ word-beam-search kurulu")
except ImportError:
    print("⚠️  word-beam-search kurulamadı — WBS atlanacak, greedy+trigram kullanılacak")

In [ ]:
# Hücre 3: IAM dataset path'ini bul
import os, subprocess

print("=== /kaggle/input altındaki tüm datasetler ===")
!ls /kaggle/input/
print()
print("=== words.txt aranıyor (maxdepth 6) ===")
result = subprocess.run(
    ["find", "/kaggle/input", "-name", "words.txt", "-maxdepth", "6"],
    capture_output=True, text=True
)
found_words_txts = [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]
for p in found_words_txts:
    print(" ", p)
if not found_words_txts:
    print("  (bulunamadı — IAM dataset eklendi mi?)")

IAM_WORDS_TXT = None
IAM_WORDS_DIR = None

for wt in found_words_txts:
    candidate_dir = os.path.join(os.path.dirname(wt), "words")
    if os.path.isdir(candidate_dir):
        IAM_WORDS_TXT = wt
        IAM_WORDS_DIR = candidate_dir
        break

if IAM_WORDS_TXT and IAM_WORDS_DIR:
    print(f"\n✓ IAM words.txt : {IAM_WORDS_TXT}")
    print(f"✓ IAM words/    : {IAM_WORDS_DIR}")
else:
    print("\n⚠️  words/ klasörü bulunamadı. Mevcut words.txt paths:")
    for p in found_words_txts:
        print(f"  {p}  →  words/: {os.path.isdir(os.path.join(os.path.dirname(p), 'words'))}")
    raise RuntimeError("IAM dataset bulunamadı — Add Data sekmesinden ekle")

In [ ]:
# Hücre 4: Eğitimi başlat (~3-3.5 saat)
# B1: val_wa early stopping (model_v3.py), B2: epochs=80, patience=20
MODEL_DIR = "/kaggle/working/Model_aachen_v3_2"

!python cloud/v3_2_train.py \
    --epochs 80 \
    --batch 128 \
    --lr 7e-4 \
    --patience 20 \
    --model-dir {MODEL_DIR} \
    --iam-words {IAM_WORDS_TXT} \
    --iam-root {IAM_WORDS_DIR}

In [ ]:
# Hücre 5: Sonuçları oku ve özetle
import json, os

results_path = "/kaggle/working/results/v3_2_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        r = json.load(f)

    print("=" * 55)
    print(" V3.2 SONUÇLARI")
    print("=" * 55)
    print(f" Greedy+Trigram WA : {r.get('greedy_trigram_wa_pct', 'N/A'):.2f}%")
    if r.get('wbs_wa_pct'):
        print(f" Word Beam Search  : {r['wbs_wa_pct']:.2f}%  ← BEST")
        ci = r.get('wbs_wilson_95ci_pct', [0, 0])
        print(f" WBS Wilson 95% CI : [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    else:
        ci = r.get('greedy_trigram_wilson_95ci_pct', [0, 0])
        print(f" Wilson 95% CI     : [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    print(f" Best Val WA       : {r['training_best_val_wa_pct']:.2f}%")
    print(f" N samples         : {r['n_samples']:,}")

    mn = r.get('mcnemar_vs_v3_base', {})
    if mn:
        print(f"\n McNemar (V3-base vs V3.2)")
        print(f" Baseline WA  : {mn.get('baseline_wa_pct', 'N/A'):.2f}%")
        print(f" Delta pp     : {mn.get('delta_pp', 'N/A'):+.2f}pp")
        print(f" p-value      : {mn.get('mcnemar_p', 'N/A'):.2e}")
        print(f" p<.01        : {'YES ✓' if mn.get('significant_p01') else 'NO'}")
    print("=" * 55)
    best_wa = r.get('wbs_wa_pct') or r.get('greedy_trigram_wa_pct', 0)
    best_ci = r.get('wbs_wilson_95ci_pct') or r.get('greedy_trigram_wilson_95ci_pct', [0, 0])
    print("\n--- Arkadaşına gönder ---")
    print(f"Test WA  : {best_wa:.2f}%")
    print(f"Wilson CI: [{best_ci[0]:.2f}%, {best_ci[1]:.2f}%]")
    if mn:
        print(f"McNemar p: {mn.get('mcnemar_p', 'N/A'):.2e}")
        print(f"Delta pp : {mn.get('delta_pp', 'N/A'):+.2f}pp")
    prev_wa = 84.5448
    delta = best_wa - prev_wa
    print(f"\n v3_augmented (84.54%) ile delta: {delta:+.2f}pp")
else:
    print(f"⚠️ {results_path} bulunamadı — eğitim tamamlandı mı?")